In [5]:
print("MM21B030")

MM21B030


In [1]:
from seq2seq_model import Seq2SeqModel 
from load_data import get_data_loaders 
import torch
import torch.nn as nn

# Initialize data loaders and model
train_loader, dev_loader, test_loader, char_to_idx_devanagari, char_to_idx_latin = get_data_loaders()
input_vocab_size = len(char_to_idx_devanagari)
output_vocab_size = len(char_to_idx_latin)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model = Seq2SeqModel(
    input_vocab_size=input_vocab_size,
    output_vocab_size=output_vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    cell_type="lstm",
    num_layers_encoder=1,
    num_layers_decoder=1,
    dropout=0.2,
    device=device,
    beam_size=1
).to(device)

criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters())

# Training loop
for epoch in range(5):
    model.train()
    total_loss = 0
    
    for src, trg in train_loader:
        # Move data to device
        src = src.to(device)
        trg = trg.to(device)
        
        # Forward pass
        output = model(src, trg[:, :-1])  # Teacher forcing with shifted target
        
        # Calculate loss (ignore padding)
        loss = criterion(output.reshape(-1, output_vocab_size), 
                        trg[:, 1:].reshape(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}')

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for src, trg in dev_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            loss = criterion(output.reshape(-1, output_vocab_size), 
                          trg[:, 1:].reshape(-1))
            val_loss += loss.item()
        print(f'Validation Loss: {val_loss/len(dev_loader):.4f}')

Epoch 1, Loss: 0.9887
Validation Loss: 0.5313
Epoch 2, Loss: 0.4663
Validation Loss: 0.3931
Epoch 3, Loss: 0.3712
Validation Loss: 0.3325
Epoch 4, Loss: 0.3250
Validation Loss: 0.3047
Epoch 5, Loss: 0.2982
Validation Loss: 0.2899


# **Architecture_sweep**

In [ ]:
sweep_config = {
    'name': 'architecture_search',
    'method': 'grid',
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {'values': [5]},
        'embedding_dim': {'values': [64,128]},
        'hidden_dim': {'values': [64, 128, 256]},
        'cell_type': {'values': ['LSTM', 'GRU', 'RNN']},
        'num_layers_encoder': {'values': [1, 2]},
        'num_layers_decoder': {'values': [1, 2]},
        'dropout': {'values': [0.2, 0.3]},
        'batch_size': {'values': [32]},
        'beam_size': {'values': [1]},
        'learning_rate': {'values' : [10e-3, 5e-4, 10e-4]}
        }
}

# **Fine_search Sweep**

In [ ]:
sweep_config = {
    'name': 'fine_search',
    'method': 'grid',
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {'values': [10, 15]},
        'embedding_dim': {'values': [64]},
        'hidden_dim': {'values': [256]},
        'cell_type': {'values': ['LSTM']},
        'num_layers_encoder': {'values': [1]},
        'dropout': {'values': [0.25, 0.35]},
        'batch_size': {'values': [32, 64]},
        'beam_size': {'values': [1, 2]},
        'learning_rate': {'values' : [10e-4]}
        }
}


# **Testing Best Model**

In [3]:
import load_data 
import importlib

# Reload both modules

importlib.reload(load_data)

from seq2seq_model import Seq2SeqModel 
from load_data import get_data_loaders 
import torch
import torch.nn as nn

# Load data
train_loader, dev_loader, test_loader, char_to_idx_latin, char_to_idx_devanagari = get_data_loaders()

# Input is Latin, Output is Devanagari
input_vocab_size = len(char_to_idx_latin)
output_vocab_size = len(char_to_idx_devanagari)

# Index maps
idx_to_latin = {v: k for k, v in char_to_idx_latin.items()}
idx_to_devanagari = {v: k for k, v in char_to_idx_devanagari.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = Seq2SeqModel(
    input_vocab_size=input_vocab_size,
    output_vocab_size=output_vocab_size,
    embedding_dim=64,
    hidden_dim=256,
    cell_type="lstm",
    num_layers_encoder=1,
    num_layers_decoder=1,
    dropout=0.35,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters())

# Training loop
best_val_loss = float('inf')
for epoch in range(15):
    model.train()
    train_loss = 0
    
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)

        output = model(src, trg[:, :-1])  # teacher forcing
        loss = criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, trg in dev_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            val_loss += criterion(
                output.reshape(-1, output_vocab_size),
                trg[:, 1:].reshape(-1)
            ).item()
    
    val_loss /= len(dev_loader)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss/len(train_loader):.4f}, Val Loss = {val_loss:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print("Saved new best model")

# Load best model
model.load_state_dict(torch.load('best_model.pt'))

# Helper: Decode sequence
def decode_sequence(indices, idx_to_char):
    return ''.join([idx_to_char.get(idx, '') for idx in indices if idx not in {0, 1, 2}])

# Testing loop
model.eval()
test_loss = 0
correct = 0
total = 0
examples = []

with torch.no_grad():
    for src, trg in test_loader:
        src, trg = src.to(device), trg.to(device)
        output = model(src, trg[:, :-1])

        # Loss
        test_loss += criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        ).item()
        
        # Accuracy (with safe min length)
        _, predicted = output.max(2)
        min_len = min(predicted.size(1), trg[:, 1:].size(1))
        correct += (predicted[:, :min_len] == trg[:, 1:][:, :min_len]).sum().item()
        total += (trg[:, 1:][:, :min_len] != 0).sum().item()
        
        # Save sample predictions
        for i in range(min(3, src.size(0))):
            input_seq = decode_sequence(src[i].cpu().numpy(), idx_to_latin)
            pred_seq = decode_sequence(predicted[i].cpu().numpy(), idx_to_devanagari)
            true_seq = decode_sequence(trg[i].cpu().numpy(), idx_to_devanagari)
            examples.append((input_seq, pred_seq, true_seq))

# Report
test_loss /= len(test_loader)
accuracy = correct / total
print(f'\nTest Loss: {test_loss:.4f}, Accuracy: {accuracy:.2%}')

print("\nExample Predictions:")
for i, (input_seq, pred_seq, true_seq) in enumerate(examples[:10]):
    status = "✓" if pred_seq == true_seq else "✗"
    print(f"{i+1}. Input: {input_seq}")
    print(f"   Pred: {pred_seq}")
    print(f"   True: {true_seq} ({status})\n")

# Save all test predictions to a file
with open("predictions_vanilla.tsv", "w", encoding="utf-8") as f:
    f.write("latin\tpredicted\tground_truth\n")
    for input_seq, pred_seq, true_seq in examples:
        f.write(f"{input_seq}\t{pred_seq}\t{true_seq}\n")
print("\nSaved all predictions to predictions_vanilla.tsv")

Epoch 1: Train Loss = 1.8833, Val Loss = 0.9049
Saved new best model
Epoch 2: Train Loss = 0.8193, Val Loss = 0.6421
Saved new best model
Epoch 3: Train Loss = 0.6145, Val Loss = 0.5423
Saved new best model
Epoch 4: Train Loss = 0.5211, Val Loss = 0.5032
Saved new best model
Epoch 5: Train Loss = 0.4626, Val Loss = 0.4801
Saved new best model
Epoch 6: Train Loss = 0.4202, Val Loss = 0.4636
Saved new best model
Epoch 7: Train Loss = 0.3872, Val Loss = 0.4617
Saved new best model
Epoch 8: Train Loss = 0.3613, Val Loss = 0.4581
Saved new best model
Epoch 9: Train Loss = 0.3384, Val Loss = 0.4539
Saved new best model
Epoch 10: Train Loss = 0.3192, Val Loss = 0.4566
Epoch 11: Train Loss = 0.3010, Val Loss = 0.4615
Epoch 12: Train Loss = 0.2876, Val Loss = 0.4728
Epoch 13: Train Loss = 0.2760, Val Loss = 0.4692
Epoch 14: Train Loss = 0.2626, Val Loss = 0.4668
Epoch 15: Train Loss = 0.2512, Val Loss = 0.4781

Test Loss: 0.4545, Accuracy: 86.18%

Example Predictions:
1. Input: ank
   Pred: अंक

In [2]:
# Save all test predictions to a file
with open("predictions_vanilla.tsv", "w", encoding="utf-8") as f:
    f.write("latin\tpredicted\tground_truth\n")
    for input_seq, pred_seq, true_seq in examples:
        f.write(f"{input_seq}\t{pred_seq}\t{true_seq}\n")
print("\nSaved all predictions to predictions_vanilla.tsv")



Saved all predictions to predictions_vanilla.tsv


# **Attention_sweep**

In [9]:
import torch
import importlib
import attention
import load_data
import torch.nn as nn

# Reload both modules
importlib.reload(attention)
importlib.reload(load_data)

# Re-import after reload
from attention import DualAttentionSeq2Seq
from load_data import get_data_loaders

# Load data (Latin → Devanagari)
train_loader, dev_loader, test_loader, char_to_idx_latin, char_to_idx_devanagari = get_data_loaders()

# Vocabulary info
input_vocab_size = len(char_to_idx_latin)
output_vocab_size = len(char_to_idx_devanagari)
idx_to_input = {v: k for k, v in char_to_idx_latin.items()}
idx_to_output = {v: k for k, v in char_to_idx_devanagari.items()}

# Device
device = torch.device("cuda")

# Model
model = DualAttentionSeq2Seq(
    input_vocab_size=input_vocab_size,
    output_vocab_size=output_vocab_size,
    embedding_dim=64,
    hidden_dim=64,
    num_layers=2,
    num_heads=2,
    dropout=0.2,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters())

# Training loop
best_val_loss = float('inf')
for epoch in range(15):
    model.train()
    train_loss = 0
    
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)

        output = model(src, trg[:, :-1])  # teacher forcing
        loss = criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, trg in dev_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            val_loss += criterion(
                output.reshape(-1, output_vocab_size),
                trg[:, 1:].reshape(-1)
            ).item()
    
    val_loss /= len(dev_loader)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss/len(train_loader):.4f}, Val Loss = {val_loss:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_attention_model.pt')
        print("Saved new best model")

# Load best model
model.load_state_dict(torch.load('best_attention_model.pt'))

# Helper: Decode sequence
def decode_sequence(indices, idx_to_char):
    return ''.join([idx_to_char.get(idx, '') for idx in indices if idx not in {0, 1, 2}])

# Testing loop
model.eval()
test_loss = 0
correct = 0
total = 0
examples = []

with torch.no_grad():
    for src, trg in test_loader:
        src, trg = src.to(device), trg.to(device)
        output = model(src, trg[:, :-1])

        # Loss
        test_loss += criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        ).item()
        
        # Accuracy (with safe min length)
        _, predicted = output.max(2)
        min_len = min(predicted.size(1), trg[:, 1:].size(1))
        correct += (predicted[:, :min_len] == trg[:, 1:][:, :min_len]).sum().item()
        total += (trg[:, 1:][:, :min_len] != 0).sum().item()
        
        # Save sample predictions
        for i in range(min(3, src.size(0))):
            input_seq = decode_sequence(src[i].cpu().numpy(), idx_to_latin)
            pred_seq = decode_sequence(predicted[i].cpu().numpy(), idx_to_devanagari)
            true_seq = decode_sequence(trg[i].cpu().numpy(), idx_to_devanagari)
            examples.append((input_seq, pred_seq, true_seq))

# Report
test_loss /= len(test_loader)
accuracy = correct / total
print(f'\nTest Loss: {test_loss:.4f}, Accuracy: {accuracy:.2%}')

print("\nExample Predictions:")
for i, (input_seq, pred_seq, true_seq) in enumerate(examples[:10]):
    status = "✓" if pred_seq == true_seq else "✗"
    print(f"{i+1}. Input: {input_seq}")
    print(f"   Pred: {pred_seq}")
    print(f"   True: {true_seq} ({status})\n")

# Save all test predictions to a file
with open("predictions_attention.tsv", "w", encoding="utf-8") as f:
    f.write("latin\tpredicted\tground_truth\n")
    for input_seq, pred_seq, true_seq in examples:
        f.write(f"{input_seq}\t{pred_seq}\t{true_seq}\n")
print("\nSaved all predictions to predictions_attention.tsv")


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
